In [20]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

CUDA available: True
GPU: Tesla T4


# 04D — Portable local / Colab setup and experiments

Default: **non-training smoke test**. Locally, use the current checkout and a pinned Python environment; set `EDGEAI_DATA_ROOT` before importing tools to use an external EdgeAI storage tree. Colab mounts Drive, clones or fast-forward pulls the GitHub repository, and installs its pinned runtime requirements. Code executes in the connected kernel's filesystem.

Upload the complete staged `EdgeAI` tree to `/content/drive/MyDrive/EdgeAI` and commit/push local code fixes before the first Colab run. Never run 00–03B to repair a missing cache. This notebook does not push code or upload data.


In [21]:
from pathlib import Path, PureWindowsPath
import os, subprocess, sys
try:
    from google.colab import drive
except ImportError:
    drive = None
IN_COLAB = drive is not None

REPO_URL = "https://github.com/ancaranovik/Mosquito_Wingbeat_v1.git"
if IN_COLAB:
    drive.mount("/content/drive")
    assert REPO_URL.startswith("https://github.com/") and "@" not in REPO_URL
    repo_hint = os.environ.get("EDGEAI_REPO_ROOT", "/content/edge-ai")
    if PureWindowsPath(repo_hint).drive:
        raise ValueError("Colab needs a runtime checkout, not a Windows repository path")
    repo = Path(repo_hint).resolve()
    if not repo.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
    else:
        origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=repo, text=True).strip()
        assert origin == REPO_URL, "Existing checkout has a different origin"
        assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip(), "Preserve uncommitted runtime changes before pulling"
        subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
else:
    hint = Path(os.environ.get("EDGEAI_REPO_ROOT", Path.cwd())).resolve()
    repo = next((p for p in [hint, *hint.parents] if (p / "tools/project_paths.py").is_file()), None)
    if repo is None:
        raise RuntimeError("Set EDGEAI_REPO_ROOT to your local checkout")
os.environ["EDGEAI_REPO_ROOT"] = str(repo)
os.chdir(repo)
sys.path.insert(0, str(repo / "tools"))
sys.dont_write_bytecode = True
print("Mode:", "COLAB" if IN_COLAB else "LOCAL")
print("Checkout:", repo)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Mode: COLAB
Checkout: /content/edge-ai


In Colab, install the pinned cache-consumer and notebook dependencies. Locally, use the already configured environment without installing packages. Verification and smoke testing use fresh child Python processes. Configure private GitHub authentication interactively; never embed credentials here.


In [22]:
from colab_bootstrap import configure
paths = configure(install=IN_COLAB)


Repository: /content/edge-ai
Data: /content/drive/MyDrive/EdgeAI
Immutable baseline: /content/drive/MyDrive/EdgeAI/fixed/baseline/stage04
Future results: /content/drive/MyDrive/EdgeAI/experiments/<experiment_id>


Verify all frozen identities and eight saved baseline runs. This reads saved predictions and hashes; it does not evaluate a trained model on TEST again or regenerate any data.

In [23]:
subprocess.run([sys.executable, "-B", "tools/verify_portability.py"], cwd=repo, check=True)


CompletedProcess(args=['/usr/bin/python3', '-B', 'tools/verify_portability.py'], returncode=0)

Colab requires CUDA and reports the GPU name before eight synthetic forward checks. Local mode uses CPU. Neither mode constructs an optimizer or starts training.


In [24]:
device = "cuda" if IN_COLAB else "cpu"
subprocess.run([sys.executable, "-B", "tools/experiment_runner.py", "smoke", "--device", device], cwd=repo, check=True)
print("Future outputs:", paths.EXPERIMENT_ROOT / "<experiment_id>")


Future outputs: /content/drive/MyDrive/EdgeAI/experiments/<experiment_id>


## Một experiment = 8 cấu hình (mặc định chưa train)

Cell dưới là nơi đặt tên: `EXPERIMENT_NAME`. Một lượt chạy lần lượt bốn model
`small_cnn`, `ds_cnn`, `tc_resnet8`, `cnn_lstm` với 03A, rồi với 03B.
Không cần chạy training trong 04A/04B. 04C chỉ đọc và so sánh kết quả đã lưu.

Nhập tên mới bắt đầu bằng `exp_` và chỉ dùng chữ, số, `_`, `-`.
`EXPERIMENT_DESCRIPTION` ghi mục đích và thay đổi so với lần trước.
Sau khi các kiểm tra phía trên PASS, chủ động đổi `RUN_TRAINING` thành `True` để chạy.
Giữ `False` trong bản notebook commit/push. Nếu mở notebook trong VS Code local với
kernel Colab, thay đổi điều khiển ở bản local không sửa checkout runtime. Nếu mở
file ngay trong checkout runtime, chạy bản notebook điều khiển ở ngoài checkout để
tránh làm bẩn Git; runner yêu cầu code/config đã commit và checkout sạch.

Kết quả lưu vào `MyDrive/EdgeAI/experiments/<EXPERIMENT_NAME>/`: config thực tế,
Git commit, runtime/GPU, protocol/params, từng checkpoint, history, predictions,
metrics, confusion matrix, log và bảng tổng hợp 8 cấu hình. Tên đã tồn tại bị từ chối.
Lần lỗi giữ nguyên kết quả một phần và ghi trạng thái FAILED/INTERRUPTED; lần thử lại
phải dùng tên mới. Không ghi vào `fixed/` và không tự tạo lại cache.


In [ ]:
# CHỈNH TÊN EXPERIMENT TẠI ĐÂY; mỗi lần train dùng tên chưa tồn tại.
EXPERIMENT_NAME = "First Run with Defult Params"
EXPERIMENT_DESCRIPTION = "Lần 1: chạy đủ 8 cấu hình với protocol CUDA cố định và cache đã xác thực."
RUN_TRAINING = False

EXPERIMENT_CONFIG = "configs/experiments/baseline_suite.json"
experiment_path = paths.experiment(EXPERIMENT_NAME)
print("Experiment:", EXPERIMENT_NAME)
print("Kết quả:", experiment_path)
print("Kế hoạch: 03A + 03B × small_cnn, ds_cnn, tc_resnet8, cnn_lstm = 8 cấu hình, chạy lần lượt.")
if RUN_TRAINING:
    subprocess.run([
        sys.executable, "-u", "-B", "tools/experiment_runner.py", "train",
        "--config", EXPERIMENT_CONFIG, "--experiment-id", EXPERIMENT_NAME,
        "--description", EXPERIMENT_DESCRIPTION,
    ], cwd=repo, check=True)
else:
    print("Training disabled. Đổi RUN_TRAINING thành True khi bạn muốn bắt đầu đủ 8 cấu hình.")


Experiment: exp_001_full_suite
Kết quả: /content/drive/MyDrive/EdgeAI/experiments/exp_001_full_suite
Kế hoạch: 03A + 03B × small_cnn, ds_cnn, tc_resnet8, cnn_lstm = 8 cấu hình, chạy lần lượt.
Training disabled. Đổi RUN_TRAINING thành True khi bạn muốn bắt đầu đủ 8 cấu hình.


## Xem kết quả sau khi đủ 8 cấu hình hoàn tất

Cell này chỉ đọc/xác thực kết quả; không train hoặc chạy lại TEST. Trong 04C, đặt cùng `EXPERIMENT_NAME` để xem history và confusion matrix của cả 8 cấu hình.


In [26]:
if RUN_TRAINING:
    from experiment_review import compare_saved_results
    from IPython.display import display
    display(compare_saved_results(EXPERIMENT_NAME))
else:
    print("Để xem lần đã chạy, mở 04C và nhập EXPERIMENT_NAME tương ứng; không cần bật training.")


Để xem lần đã chạy, mở 04C và nhập EXPERIMENT_NAME tương ứng; không cần bật training.
